**参考答案版**。请先独立完成 `lessons/` 里的练习,再来对照。

# Day 23 · 评测集审计：入库检查、饱和与退役

**心理大模型评测与 Agent** · 评测方法学 · 建议用时 3 到 4 小时

**今天的目标**

- 对一个心理健康评测集做题目分析，标出过易、过难和区分度低的题
- 检验 rubric 维度之间的冗余
- 写出可操作的饱和与退役判据
- 说明为什么审计不能由评测集的制作方自己完成

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [ ]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 题目分析

In [ ]:
from course.audit import item_analysis, cronbach_alpha, saturation_check
from course.data import simulate_vignette_matrix
u, theta, a, b, items = simulate_vignette_matrix(80, 40, seed=23)
u[:, 3] = 1                                                     # 一道所有模型都能做对的题
u[:, 7] = (np.random.default_rng(0).random(80) < 0.5)           # 一道与能力无关的题(例如评分标准有误)
tab = item_analysis(u, items.item.values)
tab[tab["flags"] != ""]

**练习 23.1**：有几道题被标记？

In [ ]:
ans_23_1 = int((tab["flags"] != "").sum())
check("练习 23.1", ans_23_1, int((tab["flags"] != "").sum()))

## 2. 维度冗余：五个 rubric 维度的相关

In [ ]:
from course.data import simulate_counselor_study, DIMENSIONS
st = simulate_counselor_study(seed=23)
wide = st["human"][st["human"].dim.isin(DIMENSIONS)].groupby(["item", "system", "dim"]).score.mean().unstack("dim")
corr = wide.corr(); corr.round(2)

**练习 23.2**：除对角线外，相关最高的一对维度是哪两个？填一个排好序的列表，例如 `["D1", "D3"]`。

In [ ]:
_c = corr.where(~np.eye(5, dtype=bool)).stack()
ans_23_2 = sorted(_c.idxmax())
check("练习 23.2", sorted(ans_23_2) if ans_23_2 else ans_23_2, sorted(_c.idxmax()))

**练习 23.3**：把五个维度当作五道题，计算 Cronbach's α。

In [ ]:
ans_23_3 = cronbach_alpha(wide.values)
check("练习 23.3", ans_23_3, cronbach_alpha(wide.values), tol=1e-6)

## 3. 饱和判断

In [ ]:
scores = {"model_1": (0.97, 0.94, 0.99), "model_2": (0.96, 0.93, 0.98), "model_3": (0.95, 0.91, 0.98), "model_4": (0.81, 0.75, 0.86)}
saturation_check(scores)

**练习 23.4**：如果 model_3 的结果是 (0.90, 0.86, 0.94)，这个评测集还算饱和吗？填 `True` 或 `False`。

In [ ]:
ans_23_4 = saturation_check({**scores, "model_3": (0.90, 0.86, 0.94)})["saturated"]
check("练习 23.4", ans_23_4, saturation_check({**scores, "model_3": (0.90, 0.86, 0.94)})["saturated"])

## 口述题（录音，每题 90 秒）

1. 评测集的饱和和退役，你用什么判据？
   <details><summary>参考要点</summary>领先模型接近上限且区间重叠；多数题被所有模型做对；有污染证据；或者使用场景变化导致效度不再成立。退役要记录原因，保留版本，用于纵向比较。</details>
2. 为什么评测集入库的审计不能由制作方自己完成？
   <details><summary>参考要点</summary>制作方有动机让评测集通过，也容易对自己的设计盲点视而不见。独立审计类似同行评审和独立数据监查委员会。</details>

## 研究延伸

对已发表的心理健康 LLM 基准做一次系统的“测量审计”（构念定义、信度证据、效度证据、污染风险），可以写成一篇方法学评论。这和你的元分析经验很契合。

In [ ]:
summary()